# HoopStats Analysis Notebook

This notebook provides comprehensive analysis of basketball tracking data:
- Load and explore detection and tracking data
- Analyze player trajectories and zones
- Compute statistics on player positions and movements
- Visualize results on court and bird's eye view

## 1. Setup and Imports

In [2]:
import os
import sys
import pathlib
import pickle
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
from time import time
import importlib

# Import project modules
from courtvisionlib.functions import *
from cav.objects import BoundingBox, Object, ObjectType
import project_config as config

importlib.reload(config)
%matplotlib inline

print("All imports successful!")

ModuleNotFoundError: No module named 'courtvisionlib'

## 2. Load Configuration and Parameters

In [ ]:
# Load parameters
with open('params.json', 'r') as f:
    params = json.load(f)

print("Parameters loaded:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Extract key parameters
video_shape = tuple(params['videoShape'])
bird_eye_shape = tuple(params['birdEyeViewShape'])
camera_points = np.array(params['cameraPoints'], dtype='float32')
bird_eye_points = np.array(params['birdEyePoints'], dtype='float32')

print(f"\nVideo shape: {video_shape}")
print(f"Bird eye view shape: {bird_eye_shape}")

## 3. Load Detection and Tracking Data

In [ ]:
# Load detections
detections_path = os.path.join(config.DATA_PATH, 'detections.p')

if os.path.exists(detections_path):
    detections = pickle.load(open(detections_path, 'rb'))
    print(f"Loaded detections from {detections_path}")
    print(f"Total frames with detections: {len(detections)}")

    # Show sample
    sample_frame = list(detections.keys())[0]
    boxes, scores, classes = detections[sample_frame]
    print(f"\nSample frame {sample_frame}:")
    print(f"  Number of detections: {len(boxes)}")
    print(f"  Boxes shape: {boxes.shape}")
    print(f"  Scores shape: {scores.shape}")
    print(f"  Classes shape: {classes.shape}")
else:
    print(f"Detections file not found at {detections_path}")
    detections = {}

NameError: name 'config' is not defined

## 4. Data Processing and Statistics

In [ ]:
def create_detection_summary(detections):
    """
    Creates a summary of detections across all frames.

    Returns a dictionary with statistics about detections.
    """
    if not detections:
        return None

    frame_ids = sorted(detections.keys())
    detection_counts = []
    all_scores = []
    class_counts = defaultdict(int)

    for frame_id in frame_ids:
        boxes, scores, classes = detections[frame_id]
        detection_counts.append(len(boxes))
        all_scores.extend(scores)
        for cls in classes:
            class_counts[int(cls)] += 1

    summary = {
        'total_frames': len(frame_ids),
        'frames_with_detections': sum(1 for count in detection_counts if count > 0),
        'total_detections': sum(detection_counts),
        'avg_detections_per_frame': np.mean(detection_counts),
        'max_detections_in_frame': max(detection_counts),
        'min_detections_in_frame': min(detection_counts),
        'avg_confidence': np.mean(all_scores),
        'min_confidence': np.min(all_scores),
        'max_confidence': np.max(all_scores),
        'class_distribution': dict(class_counts)
    }

    return summary

summary = create_detection_summary(detections)
if summary:
    print("Detection Summary:")
    for key, value in summary.items():
        if key != 'class_distribution':
            print(f"  {key}: {value}")
    print(f"  Class distribution: {summary['class_distribution']}")

## 5. Visualization Functions

In [ ]:
def visualize_detections_on_frame(frame, boxes, scores, classes,
                                   confidence_threshold=0.5,
                                   figsize=(15, 10)):
    """
    Visualizes detections on a frame.

    Arguments:
        frame: Input frame (BGR)
        boxes: Detection boxes
        scores: Detection confidence scores
        classes: Detection classes
        confidence_threshold: Minimum confidence to display
        figsize: Figure size
    """
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(frame_rgb)

    # Filter by confidence
    for i, (box, score) in enumerate(zip(boxes, scores)):
        if score >= confidence_threshold:
            ymin, xmin, ymax, xmax = box

            # Convert normalized coordinates to pixel coordinates
            ymin = int(ymin * frame.shape[0])
            ymax = int(ymax * frame.shape[0])
            xmin = int(xmin * frame.shape[1])
            xmax = int(xmax * frame.shape[1])

            # Draw rectangle
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                    linewidth=2, edgecolor='g', facecolor='none')
            ax.add_patch(rect)

            # Add label
            ax.text(xmin, ymin - 5, f'{score:.2f}',
                   color='white', fontsize=8,
                   bbox=dict(facecolor='green', alpha=0.5))

    plt.title('Detections')
    plt.axis('off')
    plt.tight_layout()
    return fig, ax

print("Visualization functions defined.")

## 6. Load and Display Sample Frame

In [ ]:
# Load a sample frame
frame_image_path = './images/frame_view1.jpg'
if os.path.exists(frame_image_path):
    sample_frame_img = cv2.imread(frame_image_path)
    sample_frame_rgb = cv2.cvtColor(sample_frame_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(16, 10))
    plt.imshow(sample_frame_rgb)
    plt.title('Sample Frame from Video')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"Frame shape: {sample_frame_img.shape}")
else:
    print(f"Sample frame not found at {frame_image_path}")

## 7. Bird's Eye View Analysis

In [ ]:
# Load bird's eye view image
sky_view_path = './images/Sky View.jpg'
if os.path.exists(sky_view_path):
    sky_view_img = cv2.imread(sky_view_path)
    sky_view_rgb = cv2.cvtColor(sky_view_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(16, 10))
    plt.imshow(sky_view_rgb)
    plt.title('Bird\'s Eye View (Sky View)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"Bird's eye view shape: {sky_view_img.shape}")
else:
    print(f"Sky view not found at {sky_view_path}")

## 8. Zone Mask Analysis

## 9. Perspective Transform Analysis

In [ ]:
# Calculate perspective transform matrices
M = cv2.getPerspectiveTransform(camera_points, bird_eye_points)
M_inv = cv2.getPerspectiveTransform(bird_eye_points, camera_points)

print("Perspective Transform Matrices calculated successfully.")

## 10. Load Tracking Data

In [ ]:
def load_tracking_data(csv_path=None):
    """
    Load tracking data from CSV or create sample DataFrame.

    Expected columns: frame, zone, playerId, x, y, birdEyeX, birdEyeY,
                     timestamp, objectType, teamId, possession
    """
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"Loaded tracking data from {csv_path}")
    else:
        # Create empty DataFrame with expected columns
        df = pd.DataFrame(columns=[
            'frame', 'zone', 'playerId', 'x', 'y', 'birdEyeX', 'birdEyeY',
            'timestamp', 'objectType', 'teamId', 'possession'
        ])
        print("Created empty tracking DataFrame. Load your CSV with: df = load_tracking_data('path/to/data.csv')")

    print(f"\nDataFrame shape: {df.shape}")
    if not df.empty:
        print(f"\nDataFrame columns: {list(df.columns)}")
        print(f"\nFirst few rows:")
        print(df.head())
        print(f"\nData types:")
        print(df.dtypes)

    return df

# Load or create tracking data
tracking_df = load_tracking_data()
# To load your own data, uncomment and modify:
# tracking_df = load_tracking_data('path/to/your/tracking_data.csv')

## 11. Movement and Speed Analytics

In [ ]:
def compute_speed_metrics(df):
    """
    Compute speed and movement metrics for each player.

    Metrics:
    - Distance traveled (in pixels and bird's eye coordinates)
    - Average speed
    - Max speed
    - Acceleration
    """
    if df.empty or 'timestamp' not in df.columns:
        print("Cannot compute speed metrics - empty DataFrame or missing timestamp")
        return pd.DataFrame()

    metrics = []

    # Group by playerId
    for player_id in df['playerId'].unique():
        player_data = df[df['playerId'] == player_id].sort_values('frame').reset_index(drop=True)

        if len(player_data) < 2:
            continue

        # Extract coordinates
        x = player_data['x'].values
        y = player_data['y'].values
        timestamps = player_data['timestamp'].values

        # Compute frame-to-frame distances (camera space)
        dx = np.diff(x)
        dy = np.diff(y)
        distances_camera = np.sqrt(dx**2 + dy**2)

        # Compute time deltas
        time_deltas = np.diff(timestamps)
        time_deltas = np.where(time_deltas == 0, 1e-6, time_deltas)  # Avoid division by zero

        # Compute speeds (pixels per second)
        speeds = distances_camera / time_deltas

        # Compute accelerations
        accelerations = np.diff(speeds) / time_deltas[:-1] if len(speeds) > 1 else np.array([])

        # Bird's eye metrics
        be_x = player_data['birdEyeX'].values
        be_y = player_data['birdEyeY'].values
        be_dx = np.diff(be_x)
        be_dy = np.diff(be_y)
        distances_sky = np.sqrt(be_dx**2 + be_dy**2)

        # Summary metrics
        player_metrics = {
            'playerId': player_id,
            'teamId': player_data['teamId'].iloc[0] if 'teamId' in player_data.columns else None,
            'objectType': player_data['objectType'].iloc[0] if 'objectType' in player_data.columns else None,
            'total_frames': len(player_data),
            'total_time_seconds': player_data['timestamp'].max() - player_data['timestamp'].min(),
            'total_distance_camera': distances_camera.sum(),
            'total_distance_sky': distances_sky.sum(),
            'avg_speed_camera': speeds.mean() if len(speeds) > 0 else 0,
            'max_speed_camera': speeds.max() if len(speeds) > 0 else 0,
            'min_speed_camera': speeds.min() if len(speeds) > 0 else 0,
            'avg_speed_sky': (distances_sky / time_deltas).mean() if len(distances_sky) > 0 else 0,
            'max_speed_sky': (distances_sky / time_deltas).max() if len(distances_sky) > 0 else 0,
            'avg_acceleration': accelerations.mean() if len(accelerations) > 0 else 0,
            'max_acceleration': accelerations.max() if len(accelerations) > 0 else 0,
        }

        metrics.append(player_metrics)

    speed_df = pd.DataFrame(metrics)
    return speed_df

if not tracking_df.empty:
    speed_metrics = compute_speed_metrics(tracking_df)
    print("\n" + "="*80)
    print("SPEED AND MOVEMENT METRICS")
    print("="*80)
    if not speed_metrics.empty:
        print(speed_metrics.to_string())
    else:
        print("No speed metrics computed - check data format")
else:
    print("Tracking data is empty - load data first")

## 12. Zone Analytics

In [ ]:
def compute_zone_metrics(df):
    """
    Compute zone-based metrics for each player.

    Metrics:
    - Time spent in each zone
    - Zone visit count
    - Zone transitions
    - Most common zone
    """
    if df.empty or 'zone' not in df.columns:
        print("Cannot compute zone metrics - empty DataFrame or missing zone column")
        return pd.DataFrame()

    zone_metrics = []

    # Zone names for reference
    zone_names = {0: 'Outside', 1: 'Right 3pt', 2: 'Right 2pt', 3: 'Left 3pt', 4: 'Left 2pt'}

    # Group by playerId
    for player_id in df['playerId'].unique():
        player_data = df[df['playerId'] == player_id].sort_values('frame').reset_index(drop=True)

        if len(player_data) == 0:
            continue

        # Get timestamps
        timestamps = player_data['timestamp'].values
        zones = player_data['zone'].values

        # Zone visit counts
        unique_zones, zone_counts = np.unique(zones, return_counts=True)
        zone_frame_counts = dict(zip(unique_zones, zone_counts))

        # Time spent in each zone
        zone_times = {}
        for zone in unique_zones:
            zone_frames = player_data[player_data['zone'] == zone]
            if len(zone_frames) > 0:
                zone_start_times = zone_frames['timestamp'].min()
                zone_end_times = zone_frames['timestamp'].max()
                zone_times[zone] = zone_end_times - zone_start_times

        # Zone transitions (changes from one zone to another)
        zone_transitions = np.sum(np.diff(zones) != 0)

        # Most common zone
        most_common_zone = unique_zones[np.argmax(zone_counts)]
        most_common_zone_name = zone_names.get(most_common_zone, f'Zone {most_common_zone}')

        # Create metrics row
        player_zone_metrics = {
            'playerId': player_id,
            'teamId': player_data['teamId'].iloc[0] if 'teamId' in player_data.columns else None,
            'zone_transitions': zone_transitions,
            'most_common_zone': most_common_zone,
            'most_common_zone_name': most_common_zone_name,
            'most_common_zone_time_sec': zone_times.get(most_common_zone, 0),
            'unique_zones_visited': len(unique_zones),
        }

        # Add time and count for each zone
        for zone in range(5):  # Zones 0-4
            zone_count = zone_frame_counts.get(zone, 0)
            zone_time = zone_times.get(zone, 0)
            zone_name = zone_names.get(zone, f'Zone {zone}')

            player_zone_metrics[f'zone_{zone}_count'] = zone_count
            player_zone_metrics[f'zone_{zone}_time_sec'] = zone_time

        zone_metrics.append(player_zone_metrics)

    zone_df = pd.DataFrame(zone_metrics)
    return zone_df

if not tracking_df.empty:
    zone_metrics = compute_zone_metrics(tracking_df)
    print("\n" + "="*80)
    print("ZONE ANALYTICS")
    print("="*80)
    if not zone_metrics.empty:
        # Show summary columns
        summary_cols = ['playerId', 'teamId', 'zone_transitions', 'most_common_zone_name', 'unique_zones_visited']
        summary_cols = [col for col in summary_cols if col in zone_metrics.columns]
        print(zone_metrics[summary_cols].to_string())
    else:
        print("No zone metrics computed - check data format")
else:
    print("Tracking data is empty - load data first")

## 13. Possession and Team Analytics

In [ ]:
def compute_possession_metrics(df):
    """
    Compute possession and team-based metrics.

    Metrics:
    - Possession time and percentage
    - Team movement patterns
    - Ball proximity stats
    """
    if df.empty:
        print("Cannot compute possession metrics - empty DataFrame")
        return pd.DataFrame()

    possession_metrics = []

    # Get possession column (may or may not exist)
    has_possession = 'possession' in df.columns

    for team_id in df['teamId'].unique():
        if pd.isna(team_id):
            continue

        team_data = df[df['teamId'] == team_id]

        # Time range for this team
        if 'timestamp' in team_data.columns:
            total_time = team_data['timestamp'].max() - team_data['timestamp'].min()
        else:
            total_time = 0

        # Possession time (if possession column exists)
        if has_possession:
            possession_frames = team_data[team_data['possession'] == True]
            possession_time = (possession_frames['timestamp'].max() - possession_frames['timestamp'].min()) if len(possession_frames) > 0 else 0
            possession_pct = (len(possession_frames) / len(team_data) * 100) if len(team_data) > 0 else 0
        else:
            possession_time = 0
            possession_pct = 0

        # Player stats
        num_players = team_data['playerId'].nunique()

        # Zone distribution for team
        zone_distribution = team_data['zone'].value_counts().to_dict() if 'zone' in team_data.columns else {}

        # Average position
        avg_x = team_data['x'].mean() if 'x' in team_data.columns else None
        avg_y = team_data['y'].mean() if 'y' in team_data.columns else None
        avg_be_x = team_data['birdEyeX'].mean() if 'birdEyeX' in team_data.columns else None
        avg_be_y = team_data['birdEyeY'].mean() if 'birdEyeY' in team_data.columns else None

        team_metrics = {
            'teamId': team_id,
            'num_players': num_players,
            'total_time_seconds': total_time,
            'possession_time_seconds': possession_time,
            'possession_percentage': possession_pct,
            'avg_x': avg_x,
            'avg_y': avg_y,
            'avg_birdEyeX': avg_be_x,
            'avg_birdEyeY': avg_be_y,
            'frames_tracked': len(team_data),
        }

        possession_metrics.append(team_metrics)

    poss_df = pd.DataFrame(possession_metrics)
    return poss_df

if not tracking_df.empty:
    possession_metrics = compute_possession_metrics(tracking_df)
    print("\n" + "="*80)
    print("POSSESSION AND TEAM ANALYTICS")
    print("="*80)
    if not possession_metrics.empty:
        print(possession_metrics.to_string())
    else:
        print("No possession metrics computed")
else:
    print("Tracking data is empty - load data first")

## 14. Player Positioning and Distance Analytics

In [ ]:
def compute_positioning_metrics(df):
    """
    Compute player positioning and distance metrics.

    Metrics:
    - Court coverage (area of court covered)
    - Average distances between players
    - Spread/clustering of team
    - Heat map style coverage
    """
    if df.empty:
        print("Cannot compute positioning metrics - empty DataFrame")
        return pd.DataFrame()

    positioning_metrics = []

    for player_id in df['playerId'].unique():
        player_data = df[df['playerId'] == player_id].sort_values('frame')

        if len(player_data) == 0:
            continue

        # Get coordinates
        if 'birdEyeX' in player_data.columns and 'birdEyeY' in player_data.columns:
            x_coords = player_data['birdEyeX'].values
            y_coords = player_data['birdEyeY'].values
        elif 'x' in player_data.columns and 'y' in player_data.columns:
            x_coords = player_data['x'].values
            y_coords = player_data['y'].values
        else:
            continue

        # Court coverage metrics
        x_range = np.max(x_coords) - np.min(x_coords) if len(x_coords) > 0 else 0
        y_range = np.max(y_coords) - np.min(y_coords) if len(y_coords) > 0 else 0
        coverage_area = x_range * y_range

        # Average position
        avg_x = np.mean(x_coords)
        avg_y = np.mean(y_coords)

        # Positional variance (how spread out player movements are)
        x_variance = np.var(x_coords)
        y_variance = np.var(y_coords)
        total_variance = x_variance + y_variance

        # Distance from court center (assuming 612x433 bird's eye view)
        court_center_x = 612 / 2
        court_center_y = 433 / 2
        dist_from_center = np.sqrt((avg_x - court_center_x)**2 + (avg_y - court_center_y)**2)

        player_metrics = {
            'playerId': player_id,
            'teamId': player_data['teamId'].iloc[0] if 'teamId' in player_data.columns else None,
            'avg_x': avg_x,
            'avg_y': avg_y,
            'x_range': x_range,
            'y_range': y_range,
            'coverage_area': coverage_area,
            'position_variance': total_variance,
            'distance_from_center': dist_from_center,
            'min_x': np.min(x_coords),
            'max_x': np.max(x_coords),
            'min_y': np.min(y_coords),
            'max_y': np.max(y_coords),
        }

        positioning_metrics.append(player_metrics)

    pos_df = pd.DataFrame(positioning_metrics)
    return pos_df

if not tracking_df.empty:
    positioning_metrics = compute_positioning_metrics(tracking_df)
    print("\n" + "="*80)
    print("POSITIONING AND COVERAGE ANALYTICS")
    print("="*80)
    if not positioning_metrics.empty:
        display_cols = ['playerId', 'teamId', 'avg_x', 'avg_y', 'coverage_area', 'distance_from_center']
        display_cols = [col for col in display_cols if col in positioning_metrics.columns]
        print(positioning_metrics[display_cols].to_string())
    else:
        print("No positioning metrics computed")
else:
    print("Tracking data is empty - load data first")

## 15. Comprehensive Metrics Report

In [ ]:
def generate_comprehensive_report(df, export_csv=False):
    """
    Generate a comprehensive analytics report for all players.
    Combines speed, zone, positioning, and team metrics.

    Arguments:
        df: Tracking DataFrame
        export_csv: If True, exports metrics to CSV files
    """
    if df.empty:
        print("Cannot generate report - DataFrame is empty")
        return

    print("\n" + "="*80)
    print("COMPREHENSIVE ANALYTICS REPORT - HOOPSTATS")
    print("="*80)

    # 1. Data overview
    print("\n1. DATA OVERVIEW")
    print("-" * 80)
    print(f"Total frames tracked: {len(df)}")
    print(f"Total unique players: {df['playerId'].nunique()}")
    print(f"Total unique teams: {df['teamId'].nunique()}")
    if 'timestamp' in df.columns:
        print(f"Time range: {df['timestamp'].min():.2f}s to {df['timestamp'].max():.2f}s")
        print(f"Total duration: {df['timestamp'].max() - df['timestamp'].min():.2f}s")
    if 'zone' in df.columns:
        print(f"Zones tracked: {sorted(df['zone'].unique())}")

    # 2. Speed metrics
    print("\n2. SPEED AND MOVEMENT METRICS")
    print("-" * 80)
    speed_df = compute_speed_metrics(df)
    if not speed_df.empty:
        speed_summary = {
            'Avg player distance traveled': speed_df['total_distance_camera'].mean(),
            'Max player distance traveled': speed_df['total_distance_camera'].max(),
            'Avg player speed (camera)': speed_df['avg_speed_camera'].mean(),
            'Max player speed recorded': speed_df['max_speed_camera'].max(),
            'Avg acceleration': speed_df['avg_acceleration'].mean(),
        }
        for metric, value in speed_summary.items():
            print(f"  {metric}: {value:.2f}")

    # 3. Zone metrics
    print("\n3. ZONE ANALYTICS")
    print("-" * 80)
    zone_df = compute_zone_metrics(df)
    if not zone_df.empty:
        print(f"  Avg zone transitions per player: {zone_df['zone_transitions'].mean():.1f}")
        print(f"  Max zone transitions: {zone_df['zone_transitions'].max()}")
        print(f"  Avg unique zones visited: {zone_df['unique_zones_visited'].mean():.1f}")

    # 4. Positioning metrics
    print("\n4. POSITIONING ANALYTICS")
    print("-" * 80)
    pos_df = compute_positioning_metrics(df)
    if not pos_df.empty:
        print(f"  Avg court coverage area per player: {pos_df['coverage_area'].mean():.2f}")
        print(f"  Max court coverage area: {pos_df['coverage_area'].max():.2f}")
        print(f"  Avg distance from court center: {pos_df['distance_from_center'].mean():.2f}")

    # 5. Team metrics
    print("\n5. TEAM ANALYTICS")
    print("-" * 80)
    team_df = compute_possession_metrics(df)
    if not team_df.empty:
        for _, team in team_df.iterrows():
            print(f"\n  Team {team['teamId']}:")
            print(f"    Players: {team['num_players']}")
            print(f"    Possession: {team['possession_percentage']:.1f}%")
            print(f"    Total frames: {team['frames_tracked']}")

    # 6. Export option
    if export_csv:
        try:
            speed_df.to_csv('metrics_speed.csv', index=False)
            zone_df.to_csv('metrics_zone.csv', index=False)
            pos_df.to_csv('metrics_positioning.csv', index=False)
            team_df.to_csv('metrics_team.csv', index=False)
            print("\n✓ Metrics exported to CSV files:")
            print("  - metrics_speed.csv")
            print("  - metrics_zone.csv")
            print("  - metrics_positioning.csv")
            print("  - metrics_team.csv")
        except Exception as e:
            print(f"Error exporting CSV: {e}")

    return {
        'speed': speed_df,
        'zone': zone_df,
        'positioning': pos_df,
        'team': team_df
    }

# Generate comprehensive report
if not tracking_df.empty:
    all_metrics = generate_comprehensive_report(tracking_df, export_csv=False)
    # Change export_csv=True to save metrics to CSV files
else:
    print("Load tracking data first to generate report")

## 16. Visualization and Heatmaps

In [ ]:
def plot_court_heatmap(df, player_id=None, team_id=None):
    """
    Plot a heatmap of player positions on the bird's eye view court.

    Arguments:
        df: Tracking DataFrame
        player_id: If specified, plot only this player
        team_id: If specified, plot only this team
    """
    if df.empty:
        print("Cannot plot heatmap - empty DataFrame")
        return

    # Filter data
    plot_df = df.copy()
    if player_id is not None:
        plot_df = plot_df[plot_df['playerId'] == player_id]
    if team_id is not None:
        plot_df = plot_df[plot_df['teamId'] == team_id]

    if plot_df.empty:
        print("No data to plot with specified filters")
        return

    # Get bird's eye coordinates
    if 'birdEyeX' in plot_df.columns and 'birdEyeY' in plot_df.columns:
        x = plot_df['birdEyeX'].values
        y = plot_df['birdEyeY'].values
    else:
        print("No bird's eye coordinates found")
        return

    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 9))

    # 2D histogram for heatmap
    h = ax.hist2d(x, y, bins=30, cmap='YlOrRd', cmin=1)

    title = "Court Heatmap"
    if player_id is not None:
        title += f" - Player {player_id}"
    if team_id is not None:
        title += f" - Team {team_id}"

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Bird\'s Eye X')
    ax.set_ylabel('Bird\'s Eye Y')
    plt.colorbar(h[3], ax=ax, label='Frequency')
    plt.tight_layout()
    plt.show()

def plot_speed_over_time(df, player_id):
    """
    Plot speed over time for a specific player.
    """
    player_data = df[df['playerId'] == player_id].sort_values('frame')

    if len(player_data) < 2:
        print(f"Not enough data for player {player_id}")
        return

    x = player_data['x'].values
    y = player_data['y'].values
    timestamps = player_data['timestamp'].values

    # Compute speeds
    dx = np.diff(x)
    dy = np.diff(y)
    distances = np.sqrt(dx**2 + dy**2)
    time_deltas = np.diff(timestamps)
    time_deltas = np.where(time_deltas == 0, 1e-6, time_deltas)
    speeds = distances / time_deltas

    # Plot
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(timestamps[:-1], speeds, linewidth=1.5, color='steelblue')
    ax.set_xlabel('Timestamp (s)')
    ax.set_ylabel('Speed (pixels/s)')
    ax.set_title(f'Speed Over Time - Player {player_id}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_zone_distribution(df, player_id=None, team_id=None):
    """
    Plot zone distribution for a player or team.
    """
    plot_df = df.copy()
    if player_id is not None:
        plot_df = plot_df[plot_df['playerId'] == player_id]
    if team_id is not None:
        plot_df = plot_df[plot_df['teamId'] == team_id]

    if plot_df.empty or 'zone' not in plot_df.columns:
        print("No zone data to plot")
        return

    zone_counts = plot_df['zone'].value_counts().sort_index()
    zone_names = {0: 'Outside', 1: 'Right 3pt', 2: 'Right 2pt', 3: 'Left 3pt', 4: 'Left 2pt'}

    fig, ax = plt.subplots(figsize=(10, 6))
    zones = [zone_names.get(z, f'Zone {z}') for z in zone_counts.index]
    ax.bar(zones, zone_counts.values, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_ylabel('Frame Count')
    ax.set_title('Zone Distribution')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

print("Visualization functions defined.")

## 17. Usage Examples

### Metrics Computed:

**Speed Metrics:**
- Total distance traveled (camera & bird's eye view)
- Average/Max/Min speed
- Acceleration metrics

**Zone Metrics:**
- Time spent in each zone (Right 3pt, Right 2pt, Left 3pt, Left 2pt, Outside)
- Zone transition count
- Most common zone
- Unique zones visited

**Positioning Metrics:**
- Court coverage area
- Position variance
- Distance from court center
- Coordinate bounds (min/max x, y)

**Possession/Team Metrics:**
- Possession time and percentage
- Team movement patterns
- Average team positions
- Number of players per team

**How to use:**

```python
# Load your CSV tracking data
tracking_df = load_tracking_data('path/to/tracking_data.csv')

# Generate full report
all_metrics = generate_comprehensive_report(tracking_df, export_csv=True)

# Visualize individual players
plot_court_heatmap(tracking_df, player_id='player1')
plot_speed_over_time(tracking_df, player_id='player1')
plot_zone_distribution(tracking_df, player_id='player1')

# Or visualize by team
plot_court_heatmap(tracking_df, team_id='team1')
plot_zone_distribution(tracking_df, team_id='team1')
```